# Steps to do before running the notebook:

## 1. Add the dataset
Add the dataset to the notebook:  
https://www.kaggle.com/datasets/chilukalaamrutha/raw-data  

---

## 2. Set your Hugging Face token

### For Linux / Mac / Git Bash:
```
export HF_TOKEN=your_token_here
```

### For Windows (Command Prompt):
```
set HF_TOKEN=your_token_here
```

### For Kaggle:

1. Go to **Kaggle → Settings → Secrets**  
2. Click **"Add a new secret"**  
3. Enter:
   - **Name:** `HF_TOKEN`  
   - **Value:** your Hugging Face token  

---

*(Alternatively, you can enter the token when prompted while running the notebook.)*

In [ ]:
########################################
# SECTION 1 — Imports & Config
########################################

!pip install -q transformers accelerate bitsandbytes sentencepiece

import os
import torch
import time
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

In [ ]:
# ==============================
# CONFIG
# ==============================

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "Hugging Face token not found. Please set HF_TOKEN as an environment variable."
    )
    
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

########################################
# SECTION 2 — LLM Engine
########################################

class LLMEngine:

    def __init__(self, model_name, hf_token):

        print("Loading Llama 3.1-8B...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            token=hf_token
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            token=hf_token
        )

        self.model.eval()
        print("Model loaded!")

    def chat(self, system_prompt, user_prompt, max_tokens=200):

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        input_ids = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)

        with torch.no_grad():
            output = self.model.generate(
                input_ids,
                max_new_tokens=max_tokens,
                temperature=0.2,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(
            output[0][input_ids.shape[-1]:],
            skip_special_tokens=True
        )

        return response.strip()

llm = LLMEngine(MODEL_NAME, HF_TOKEN)

response = llm.chat(
    "You are a helpful assistant.",
    "Explain briefly why people visit coffee shops."
)

print("=== LLM RESPONSE ===")
print(response)

In [ ]:
########################################
# SECTION 3 — UPGRADED DATASET LOADER
# (Preserves raw sentences + structure)
########################################

import json
import re
from collections import defaultdict
from tqdm import tqdm

DATA_PATH = "/kaggle/input/datasets/chilukalaamrutha/raw-data"
CITY = "nyc"

TRAIN_FILE = f"{DATA_PATH}/{CITY}_train.jsonl"
TEST_FILE = f"{DATA_PATH}/{CITY}_test.jsonl"

########################################
# Regex patterns
########################################

VISIT_PATTERN = r"(At .*? POI ID \d+ \(.*?\)\.)"
STRUCT_PATTERN = r"POI ID (\d+) \((.*?), ([\-\d\.]+),([\-\d\.]+)\)"

########################################
# Parse structured trajectory
########################################

def parse_structured_trajectory(text):

    matches = re.findall(STRUCT_PATTERN, text)

    trajectory = []

    for poi_id, category, lat, lon in matches:

        trajectory.append({
            "poi": int(poi_id),
            "category": category.strip(),
            "lat": float(lat),
            "lon": float(lon)
        })

    return trajectory

########################################
# Extract raw visit sentences
########################################

def extract_raw_sentences(text):

    return re.findall(VISIT_PATTERN, text)

########################################
# Dataset class
########################################

class TrajectoryDataset:

    def __init__(self, train_file, test_file):

        print("Loading datasets...\n")

        self.train_samples = self.load_file(train_file)
        self.test_samples = self.load_file(test_file)

        print(f"\nTrain samples: {len(self.train_samples)}")
        print(f"Test samples: {len(self.test_samples)}")

        print("\nBuilding training statistics...")
        self.build_training_statistics()

        print("\nDataset ready!")

    ########################################
    # Load JSONL file
    ########################################

    def load_file(self, file_path):

        samples = []

        with open(file_path, "r") as f:

            for line in tqdm(f, desc=f"Reading {file_path.split('/')[-1]}"):

                
                try:
                    data = json.loads(line)
                except:
                    continue

                system_text = data["messages"][0]["content"]
                user_text = data["messages"][1]["content"]
                assistant_text = data["messages"][2]["content"]

                ################################
                # Extract user id
                ################################

                user_match = re.search(r"user_(\d+)", user_text)
                user_id = int(user_match.group(1)) if user_match else -1

                ################################
                # Extract BOTH representations
                ################################

                structured = parse_structured_trajectory(user_text)
                raw_sentences = extract_raw_sentences(user_text)

                if len(structured) == 0:
                    continue

                ################################
                # Extract ground truth
                ################################

                gt_match = re.search(r"(\d+)", assistant_text)
                ground_truth = int(gt_match.group(1)) if gt_match else None

                ################################
                # Store enriched sample
                ################################

                samples.append({
                    "user_id": user_id,

                    # Structured view (for retrieval)
                    "trajectory": structured,

                    # Raw text view (for LLM)
                    "raw_sentences": raw_sentences,
                    "original_user_text": user_text,
                    "system_text": system_text,
                    "assistant_text": assistant_text,

                    "ground_truth": ground_truth
                })

        return samples

    ########################################
    # Build retrieval statistics
    ########################################

    def build_training_statistics(self):

        self.transition_graph = defaultdict(lambda: defaultdict(int))
        self.poi_metadata = {}
        self.user_histories = defaultdict(list)
        self.trajectory_lengths = []

        for sample in tqdm(self.train_samples, desc="Building stats"):

            user_id = sample["user_id"]
            traj = sample["trajectory"]

            self.trajectory_lengths.append(len(traj))
            self.user_histories[user_id].extend(traj)

            ################################
            # Transition graph
            ################################

            for i in range(len(traj) - 1):

                src = traj[i]["poi"]
                dst = traj[i + 1]["poi"]

                self.transition_graph[src][dst] += 1

            ################################
            # POI metadata
            ################################

            for step in traj:

                poi = step["poi"]

                if poi not in self.poi_metadata:

                    self.poi_metadata[poi] = {
                        "category": step["category"],
                        "lat": step["lat"],
                        "lon": step["lon"]
                    }

        print(f"Unique users: {len(self.user_histories)}")
        print(f"Unique POIs: {len(self.poi_metadata)}")

########################################
# Initialize dataset
########################################

dataset = TrajectoryDataset(TRAIN_FILE, TEST_FILE)

print("\nExample enriched sample:\n")
print({
    "user": dataset.train_samples[0]["user_id"],
    "structured_len": len(dataset.train_samples[0]["trajectory"]),
    "raw_sentence_len": len(dataset.train_samples[0]["raw_sentences"])
})


In [ ]:
########################################
# SECTION 4 — PURE LLM PROFILER AGENT (ROBUST + FAST)
########################################

import random
import json
import re
from tqdm import tqdm

USE_SUBSET = True
TARGET_USERS = 50
EVAL_SAMPLES = 50
DEBUG_PROFILES = 3


########################################
# Deterministic user selection
########################################

def select_user_subset(dataset, target_users):

    user_ids = sorted(dataset.user_histories.keys())
    selected = user_ids[:target_users]

    print("\nDeterministic user selection:")
    print("Selected users:", selected)

    return set(selected)


########################################
# Robust JSON extractor
########################################

def extract_full_json(text):

    print("\n[Profiler] Raw LLM output:")
    print(text)

    # Remove markdown fences
    text = text.replace("```json", "").replace("```", "")

    # Cut at termination marker
    if "END_JSON" in text:
        text = text.split("END_JSON")[0]

    # Find JSON boundaries
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        print("[Profiler] ❌ JSON boundaries not found")
        return None

    json_str = text[start:end+1]

    try:
        data = json.loads(json_str)
        print("[Profiler] ✅ JSON parsed successfully")
        return data
    except Exception as e:
        print("[Profiler] ❌ JSON parse error:", e)
        return None


########################################
# PURE LLM PROFILER AGENT
########################################

class ProfilerAgent:

    def __init__(self, dataset, llm):

        self.dataset = dataset
        self.llm = llm
        self.user_profiles = {}
        self.debug_counter = 0

    ########################################
    # Build profile
    ########################################

    def profile_user(self, user_id):

        if user_id in self.user_profiles:
            return self.user_profiles[user_id]

        history = self.dataset.user_histories[user_id]

        # Condense trajectory (faster + prevents OOM)
        condensed = history[-40:]  # last 40 visits

        history_text = [
            f"POI {s['poi']} ({s['category']})"
            for s in condensed
        ]

        system_prompt = """
You are an expert behavioral analyst.

Analyze the user's mobility trajectory
and build a structured behavioral profile.

Return STRICT valid JSON only.
No explanations.
No markdown.
End your response with: END_JSON
"""

        user_prompt = f"""
Trajectory:
{history_text}

Return JSON:

{{
  "behavioral_summary": "...",
  "preferred_categories": [],
  "routine_patterns": {{}},
  "exploration_style": "...",
  "temporal_habits": {{}},
  "recommendation_strategy_hint": "..."
}}

Rules:
- Return ONLY JSON
- Close all braces
- End with END_JSON
"""

        print(f"\n[Profiler] Building profile for user {user_id}...")

        response = self.llm.chat(
            system_prompt,
            user_prompt,
            max_tokens=350
        )

        profile = extract_full_json(response)

        if profile is None:

            print("[Profiler] ⚠️ Using safe fallback profile")

            profile = {
                "behavioral_summary": "unknown",
                "preferred_categories": [],
                "routine_patterns": {},
                "exploration_style": "unknown",
                "temporal_habits": {},
                "recommendation_strategy_hint": "none"
            }

        if self.debug_counter < DEBUG_PROFILES:
            print("\n[Profiler] Final structured profile:")
            print(json.dumps(profile, indent=2))
            self.debug_counter += 1

        self.user_profiles[user_id] = profile
        return profile

    ########################################
    # Build all profiles
    ########################################

    def build_profiles(self, user_set):

        print("\nBuilding pure LLM user profiles...\n")

        for user_id in tqdm(sorted(user_set)):
            self.profile_user(user_id)

        print("\nProfiling complete!")


########################################
# BUILD PROFILES
########################################

if USE_SUBSET:
    active_users = select_user_subset(dataset, TARGET_USERS)
    print(f"\nUsing subset of {len(active_users)} users")
else:
    active_users = set(dataset.user_histories.keys())
    print(f"\nUsing ALL {len(active_users)} users")

profiler = ProfilerAgent(dataset, llm)
profiler.build_profiles(active_users)


########################################
# Example profile display
########################################

example_user = sorted(profiler.user_profiles.keys())[0]

print("\nExample LLM profile:")
print(json.dumps(profiler.user_profiles[example_user], indent=2))


########################################
# TEST SAMPLING
########################################

print("\nSampling evaluation test set (profile-aware)...")

profiled_users = set(profiler.user_profiles.keys())

profiled_test_samples = [
    s for s in dataset.test_samples
    if s["user_id"] in profiled_users
]

active_test_samples = random.sample(
    profiled_test_samples,
    min(EVAL_SAMPLES, len(profiled_test_samples))
)

print("Evaluation samples:", len(active_test_samples))
print("Profiler users:", sorted(active_users))

In [ ]:
########################################
# PROFILE-DRIVEN LLM RETRIEVAL (FULL DROP-IN)
########################################

import numpy as np
from collections import defaultdict, Counter
import json, re
from tqdm import tqdm

class ProfileDrivenRetrieval:

    def __init__(self, dataset, profiler, llm, top_k=125):

        self.dataset = dataset
        self.profiler = profiler
        self.llm = llm
        self.top_k = top_k

        # cache weights so we don't recompute every episode
        self.weight_cache = {}

        print("\nBuilding statistical evidence models...")
        self._build_models()
        print("✓ Retrieval engine ready")

    ########################################
    # Build empirical statistics (TRAIN DATA ONLY)
    ########################################

    def _build_models(self):

        # Transition probabilities
        self.transition_probs = {}

        for src, dests in self.dataset.transition_graph.items():
            total = sum(dests.values())

            if total == 0:
                continue

            self.transition_probs[src] = {
                dst: count / total
                for dst, count in dests.items()
            }

        # User preference probabilities
        self.user_poi_probs = {}

        for uid, history in self.dataset.user_histories.items():

            counter = Counter(step["poi"] for step in history)
            total = sum(counter.values())

            if total == 0:
                continue

            self.user_poi_probs[uid] = {
                poi: c / total
                for poi, c in counter.items()
            }

    ########################################
    # LLM profile interpreter (UNBIASED)
    ########################################

    def get_profile_weights(self, user_id):

        # CACHE → avoids repeated LLM calls
        if user_id in self.weight_cache:
            return self.weight_cache[user_id]

        profile = self.profiler.user_profiles[user_id]

        system_prompt = """
You are controlling a retrieval system.

Analyze the user profile and decide how much
to rely on:

- transition evidence (short-term trajectory)
- user preference evidence (long-term behavior)

Return ONLY JSON:

{
  "transition": number,
  "user": number
}

Weights must sum to 1.
Choose values based on reasoning.
"""

        user_prompt = json.dumps(profile, indent=2)

        print(f"\n[LLM] Requesting weights for user {user_id}...")

        response = self.llm.chat(system_prompt, user_prompt, max_tokens=120)

        print("[LLM] Raw response:")
        print(response)

        # SAFE DEFAULT
        weights = {"transition": 0.5, "user": 0.5}

        try:
            match = re.search(r"\{.*\}", response, re.DOTALL)

            if match:
                parsed = json.loads(match.group())

                t = float(parsed.get("transition", 0.5))
                u = float(parsed.get("user", 0.5))

                # clamp negatives
                t = max(t, 0)
                u = max(u, 0)

                total = t + u

                if total > 0:
                    weights["transition"] = t / total
                    weights["user"] = u / total

                print("[LLM] ✅ Parsed weights:", weights)

            else:
                print("[LLM] ⚠ No JSON found → using default")

        except Exception as e:
            print("[LLM] ⚠ Parse failed:", e)
            print("[LLM] Using default weights")

        print(f"[LLM] 👉 Final weights used for user {user_id}: {weights}")

        # CACHE RESULT
        self.weight_cache[user_id] = weights

        return weights

    ########################################
    # Retrieval
    ########################################

    def retrieve_candidates(self, trajectory, user_id):

        weights = self.get_profile_weights(user_id)

        scores = defaultdict(float)

        pois = [s["poi"] for s in trajectory]

        ################################
        # Transition evidence
        ################################

        last_pois = pois[-2:]
        N = len(last_pois)
        for p in last_pois:
            if p in self.transition_probs:
                for q, prob in self.transition_probs[p].items():
                    scores[q] += (prob / N) * weights["transition"]

        ################################
        # User preference evidence
        ################################

        if user_id in self.user_poi_probs:

            for poi, prob in self.user_poi_probs[user_id].items():

                scores[poi] += prob * weights["user"]

        ################################
        # Rank
        ################################

        ranked = sorted(scores, key=scores.get, reverse=True)

        return ranked[:self.top_k]


########################################
# INITIALIZE RETRIEVAL ENGINE
########################################

retrieval = ProfileDrivenRetrieval(dataset, profiler, llm)

########################################
# EVALUATION — RECALL@150
########################################

print("\nEvaluating retrieval...")

hits = 0
ranks = []

for sample in tqdm(active_test_samples):

    user = sample["user_id"]
    gt = sample["ground_truth"]
    traj = sample["trajectory"]

    candidates = retrieval.retrieve_candidates(traj, user)

    if gt in candidates:
        hits += 1
        ranks.append(candidates.index(gt) + 1)

recall = hits / len(active_test_samples)
mrr = np.mean([1/r for r in ranks]) if ranks else 0

print("\n==============================")
print("PROFILE-DRIVEN RETRIEVAL")
print("==============================")
print(f"Recall@125: {recall:.2%}")
print(f"MRR: {mrr:.4f}")
print("==============================")

In [ ]:
########################################
# SECTION — PURE DATASET-FORMAT PREFIX EPISODIC SPLIT (FINAL)
########################################

MIN_PREFIX = 8
EPISODE_STEP = 10

print("\nBuilding PURE dataset-format episodic set...")

import re
import json

visit_pattern = r"(At .*? POI ID \d+ \(.*?\)\.)"

episodic_dataset_rows = []
episodic_internal_samples = []

for sample in dataset.test_samples:

    user_id = sample["user_id"]
    raw_text = sample["original_user_text"]
    system_text = sample["system_text"]
    assistant_text = sample["assistant_text"]

    structured = sample["trajectory"]
    original_gt = sample["ground_truth"]

    visits = re.findall(visit_pattern, raw_text)
    n = len(visits)

    if n <= MIN_PREFIX + 1:
        continue

    ########################################
    # PREFIX SYNTHETIC EPISODES
    ########################################

    for cut in range(MIN_PREFIX, n - 1):

        if (cut - MIN_PREFIX) % EPISODE_STEP != 0:
            continue

        prefix_visits = visits[:cut]
        prefix_struct = structured[:cut]

        gt_match = re.search(r"POI ID (\d+)", visits[cut])
        if not gt_match:
            continue

        gt = int(gt_match.group(1))

        user_prompt = (
            f"<historical trajectory of user_{user_id}>\n"
            + " ".join(prefix_visits)
        )

        dataset_row = {
            "messages": [
                {"role": "system", "content": system_text},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": json.dumps({"next_poi_id": gt})}
            ]
        }

        episodic_dataset_rows.append(dataset_row)

        episodic_internal_samples.append({
            "user_id": user_id,
            "trajectory": prefix_struct,
            "ground_truth": gt,
            "episode_index": cut,
            "episode_type": "synthetic"
        })

    ########################################
    # FINAL EPISODE = ORIGINAL ROW
    ########################################

    final_row = {
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": raw_text},
            {"role": "assistant", "content": assistant_text}
        ]
    }

    episodic_dataset_rows.append(final_row)

    episodic_internal_samples.append({
        "user_id": user_id,
        "trajectory": structured,
        "ground_truth": original_gt,
        "episode_index": float("inf"),
        "episode_type": "original"
    })

########################################

active_test_samples = episodic_internal_samples

print("\nEpisodic dataset ready!")
print("Dataset-format rows:", len(episodic_dataset_rows))
print("Internal evaluation samples:", len(active_test_samples))

########################################
# DEBUG — PRINT FIRST 3 FULL DATASET ROWS
########################################

print("\n" + "="*60)
print("DEBUG: FIRST 3 FULL EPISODIC DATASET ROWS")
print("="*60)

for i, row in enumerate(episodic_dataset_rows[:3], 1):

    print(f"\nROW {i}")
    print("-"*40)

    print(json.dumps(row, indent=2))

print("\n" + "="*60)
print("END DEBUG")
print("="*60)

In [ ]:
########################################
# SECTION — LLM MEMORY AGENT (PURE MEMORY)
########################################

class LLMMemoryAgent:
    """
    Persistent episodic memory system.

    Memory models long-term user behavior.
    It evolves ONLY from trajectory data.
    Reflection is NOT used here.
    """

    def __init__(self, dataset, profiler, llm, debug=True):

        self.dataset = dataset
        self.profiler = profiler
        self.llm = llm
        self.debug = debug

        self.user_memory = {}

        print("\n✓ Pure Memory Agent initialized")

    ########################################
    # Bootstrap from training history
    ########################################

    def bootstrap_memory(self, user_id):

        history = self.dataset.user_histories.get(user_id, [])

        if not history:
            return ""

        condensed = [
            f"POI {s['poi']} ({s['category']})"
            for s in history[-20:]
        ]

        profile = self.profiler.user_profiles.get(user_id, {})

        system_prompt = """
You are a long-term behavioral memory system.

Summarize the user's stable preferences,
routines, and behavioral patterns.
"""

        user_prompt = f"""
User profile:
{profile}

Training history:
{condensed}

Write long-term behavioral memory:
"""

        memory = self.llm.chat(
            system_prompt,
            user_prompt,
            max_tokens=200
        ).strip()

        if self.debug:
            print("[Memory] Bootstrapped:", memory)

        return memory

    ########################################
    # Reset per user
    ########################################

    def reset_user(self, user_id):

        self.user_memory[user_id] = self.bootstrap_memory(user_id)

        if self.debug:
            print(f"[Memory] Reset user {user_id}")

    ########################################
    # Get memory
    ########################################

    def get_active_memory_context(self, user_id):

        memory = self.user_memory.get(user_id, "")

        if self.debug and memory:
            print("[Memory] Using:", memory)

        return memory

    ########################################
    # Update from trajectory ONLY
    ########################################

    def update_memory(self, user_id, trajectory):

        previous_memory = self.user_memory.get(user_id, "")

        condensed = [
            f"POI {s['poi']} ({s['category']})"
            for s in trajectory[-6:]
        ]

        system_prompt = """
You are a behavioral memory consolidation system.

Update the user's long-term memory
based ONLY on new trajectory evidence.
"""

        user_prompt = f"""
Previous memory:
{previous_memory}

Recent trajectory:
{condensed}

Write updated memory:
"""

        updated = self.llm.chat(
            system_prompt,
            user_prompt,
            max_tokens=200
        ).strip()

        self.user_memory[user_id] = updated

        if self.debug:
            print("[Memory] Updated:", updated)


########################################
# Initialize memory agent
########################################

memory_agent = LLMMemoryAgent(dataset, profiler, llm)

In [ ]:
########################################
# SECTION — LLM REFLECTION AGENT (DEBUG ENABLED)
########################################

from collections import defaultdict

class LLMReflectionAgent:

    def __init__(self, llm, debug=True):

        self.llm = llm
        self.reflection_memory = defaultdict(str)
        self.debug = debug

        print("\n✓ Reflection Agent initialized")

    def reset_user(self, user_id):

        self.reflection_memory[user_id] = ""

        if self.debug:
            print(f"[Reflection] Reset user {user_id}")

    def get_reflection_context(self, user_id):

        context = self.reflection_memory.get(user_id, "")

        if self.debug and context:
            print("[Reflection] Using:", context)

        return context

    def update_reflection(self, user_id, trajectory, ranked, ground_truth):

        if not ranked:
            return

        prediction = ranked[0]
        success = prediction == ground_truth

        condensed_traj = [
            f"POI {s['poi']} ({s['category']})"
            for s in trajectory[-6:]
        ]

        system_prompt = """
You are a reflection agent.
Generate short strategy improvement feedback.
"""

        user_prompt = f"""
Trajectory: {condensed_traj}
Predicted: {prediction}
GT: {ground_truth}
Success: {success}
"""

        reflection = self.llm.chat(
            system_prompt,
            user_prompt,
            max_tokens=150
        )

        self.reflection_memory[user_id] = reflection.strip()

        if self.debug:
            print("[Reflection] Updated:", reflection)


########################################
# Initialize reflection agent
########################################

reflection_agent = LLMReflectionAgent(llm)

In [ ]:
########################################
# SECTION — MEMORY + REFLECTION RERANKER
########################################

import json
import re

class LLMRerankerAgent:

    def __init__(self, llm, target_k=50):

        self.llm = llm
        self.target_k = target_k

        print("\n✓ Reranker Agent initialized")
        print(f"  Strategy: memory + reflection reranking")
        print(f"  Target k: {target_k}")

    ########################################
    # JSON extractor
    ########################################

    def extract_json(self, text):

        try:
            match = re.search(r"\[.*\]", text, re.DOTALL)
            if match:
                return json.loads(match.group())
        except:
            pass

        return []

    ########################################
    # Reranking
    ########################################

    def rerank(
        self,
        user_id,
        trajectory,
        candidates,
        memory_context,
        reflection_context   # ← NEW
    ):

        if not candidates:
            return []

        condensed_traj = [
            f"POI {s['poi']} ({s['category']})"
            for s in trajectory[-8:]
        ]

        system_prompt = """
You are a recommendation reranking agent.

Use trajectory, memory, and reflection
to rank candidates.

Return ONLY a JSON list of POI IDs.
"""

        user_prompt = f"""
User: {user_id}

Trajectory:
{condensed_traj}

Memory:
{memory_context}

Reflection:
{reflection_context}

Candidates:
{candidates[:100]}

Return top {self.target_k} ranked IDs as JSON list.
"""

        response = self.llm.chat(
            system_prompt,
            user_prompt,
            max_tokens=300
        )

        ranked = self.extract_json(response)

        if not ranked:
            return candidates[:self.target_k]

        ranked = [p for p in ranked if p in candidates]

        for p in candidates:
            if p not in ranked:
                ranked.append(p)

        return ranked[:self.target_k]


########################################
# Initialize reranker
########################################

reranker = LLMRerankerAgent(llm)     

In [ ]:
########################################
# FINAL EPISODIC PIPELINE (NO REFLECTION — ABLATION)
########################################

from collections import defaultdict
import numpy as np
import math
import pandas as pd

print("\n" + "="*60)
print("EPISODIC EVALUATION (PROFILED USERS ONLY — NO REFLECTION)")
print("="*60)

########################################
# Episode logging (UNCHANGED)
########################################

episode_logs = []

########################################
# Filter profiled users
########################################

filtered_samples = [
    s for s in active_test_samples
    if s["user_id"] in active_users
]

episodes_by_user = defaultdict(list)

for sample in filtered_samples:
    episodes_by_user[sample["user_id"]].append(sample)

########################################
# Metrics containers (UNCHANGED)
########################################

total_episodes = 0
stage1_hits = 0
stage1_ranks = []
stage2_hits = 0
stage2_ranks = []

hr5 = hr10 = ndcg5 = ndcg10 = 0

########################################
# Run episodic pipeline
########################################

for user_id, episodes in episodes_by_user.items():

    print("\n" + "="*60)
    print(f"USER {user_id} — {len(episodes)} EPISODES")
    print("="*60)

    memory_agent.reset_user(user_id)
    # reflection_agent.reset_user(user_id)  ← REMOVED

    for i, sample in enumerate(episodes, 1):

        total_episodes += 1

        print(f"\nEpisode {i}/{len(episodes)}")

        trajectory = sample["trajectory"]
        gt = sample["ground_truth"]

        ################################
        # Stage 1 — Retrieval
        ################################

        wide = retrieval.retrieve_candidates(
            trajectory,
            user_id=user_id
        )

        stage1_hit = gt in wide

        if not stage1_hit:

            print("✗ Stage 1 miss")

            episode_logs.append({
                "user": user_id,
                "episode": i,
                "stage1_hit": False,
                "stage2_hit": False,
                "rank": None,
                "memory": "",
                "reflection": ""
            })

            continue

        rank = wide.index(gt) + 1
        stage1_hits += 1
        stage1_ranks.append(rank)

        print(f"✓ Stage 1 hit (rank {rank})")

        ################################
        # Stage 2 — Memory (UNCHANGED)
        ################################

        memory_context = memory_agent.get_active_memory_context(user_id)

        print("[Pipeline] Memory ready")

        ################################
        # Reflection DISABLED
        ################################

        reflection_context = ""

        ################################
        # Stage 2 — Reranking
        ################################

        filtered = reranker.rerank(
            user_id,
            trajectory,
            wide,
            memory_context,
            reflection_context
        )

        print("Top 10 reranked:", filtered[:10])

        stage2_hit = gt in filtered

        if stage2_hit:

            rank = filtered.index(gt) + 1
            stage2_hits += 1
            stage2_ranks.append(rank)

            print(f"✓ Stage 2 hit (rank {rank})")

            if rank <= 5:
                hr5 += 1
                ndcg5 += 1 / math.log2(rank + 1)

            if rank <= 10:
                hr10 += 1
                ndcg10 += 1 / math.log2(rank + 1)

        else:
            rank = None
            print("✗ Stage 2 miss")

        ################################
        # Episode logging (UNCHANGED)
        ################################

        episode_logs.append({
            "user": user_id,
            "episode": i,
            "stage1_hit": stage1_hit,
            "stage2_hit": stage2_hit,
            "rank": rank,
            "memory": memory_context[:150],
            "reflection": ""  # reflection removed
        })

        ################################
        # Reflection update REMOVED
        ################################

        # reflection_agent.update_reflection(...) ← REMOVED

        ################################
        # Memory update (UNCHANGED)
        ################################

        memory_agent.update_memory(user_id, trajectory)


########################################
# Final metrics (UNCHANGED)
########################################

if total_episodes == 0:

    print("\nNo episodes found!")

else:

    stage1_recall = stage1_hits / total_episodes
    stage2_recall = stage2_hits / total_episodes

    stage1_mrr = np.mean([1/r for r in stage1_ranks]) if stage1_ranks else 0
    stage2_mrr = np.mean([1/r for r in stage2_ranks]) if stage2_ranks else 0

    hr5 /= total_episodes
    hr10 /= total_episodes
    ndcg5 /= total_episodes
    ndcg10 /= total_episodes

    print("\n" + "="*60)
    print("FINAL RESULTS")
    print("="*60)

    print(f"\nTotal episodes: {total_episodes}")

    print("\nStage 1 — Retrieval:")
    print(f"Recall@125: {stage1_recall:.2%}")
    print(f"MRR:        {stage1_mrr:.4f}")

    print("\nStage 2 — Reranker:")
    print(f"Recall@50:  {stage2_recall:.2%}")

    print("\nRanking Metrics (Stage 2):")
    print(f"HR@5:   {hr5:.4f}")
    print(f"HR@10:  {hr10:.4f}")
    print(f"NDCG@5: {ndcg5:.4f}")
    print(f"NDCG@10:{ndcg10:.4f}")
    print(f"MRR:    {stage2_mrr:.4f}")
    print("="*60)

########################################
# Episode-wise performance analysis
########################################

df = pd.DataFrame(episode_logs)

print("\n" + "="*60)
print("EPISODE-WISE PERFORMANCE")
print("="*60)

episode_summary = df.groupby("episode")["stage2_hit"].mean()

for ep, acc in episode_summary.items():
    print(f"Episode {ep}: {acc:.2%} success rate")

print("="*60)